## 📌 Corporate Credit Rating Prediction using ML: A Beginner-Friendly Guide

### 1️⃣ Understanding the Problem Statement & the Data

**Context**
- A corporate credit rating expresses the ability of a firm to repay its debt to creditors (nothing new for you 🤓). 
- Credit rating agencies are the entities responsible to make the assessment and give a verdict. When a big corporation from the US or anywhere in the world wants to issue a new bond it hires a credit agency to make an assessment so that investors can know how trustworthy is the company. (again you know it already 🥱)
- The assessment is based especially in the financials indicators that come from the balance sheet. Some of the most important agencies in the world are Moodys, Fitch and Standard and Poors.(I know you are sleeping now with this info 💤)

**Data 🗃️**
- A list of 2029 credit ratings issued by major agencies such as Standard and Poors to big US firms (traded on NYSE or Nasdaq) from 2010 to 2016.
- There are 30 features for every company of which 25 are financial indicators. They can be divided in:
    - Liquidity Measurement Ratios: currentRatio, quickRatio, cashRatio, daysOfSalesOutstanding
    - Profitability Indicator Ratios: grossProfitMargin, operatingProfitMargin, pretaxProfitMargin, netProfitMargin, effectiveTaxRate, returnOnAssets, returnOnEquity, returnOnCapitalEmployed
    - Debt Ratios: debtRatio, debtEquityRatio
    - Operating Performance Ratios:` assetTurnover
    - Cash Flow Indicator Ratios: operatingCashFlowPerShare, freeCashFlowPerShare, cashPerShare, operatingCashFlowSalesRatio, freeCashFlowOperatingCashFlowRatio
    - The additional features are Name, Symbol (for trading), Rating Agency Name, Date and Sector.

Data can be downloaded from: https://www.kaggle.com/datasets/agewerc/corporate-credit-rating?resource=download



### 2️⃣ Let's Dive into code and run Data Description

##### Our routine ☀️🪴🍳🧘‍♀️☕️ imports 

In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/external/corporate_rating.csv')

In [ ]:
df.head()

In [ ]:
df.describe(include='all')

### 3️⃣ Exploratory Data Analysis (EDA)
Our first step is to perform an exploratory data analysis to understand the charateristics of dataset. Here are some quesitons we will try to adress:
1. What are the dimensions of the data?
2. How do predictors relate to each other?
3. What are the classes of the data?
4. How are the predictors distributed?
5. How are the labels distributed?
6. Do we have missing values?
7. Are outliers are relevant?
8. Are there any transformations that must be done with the dataset?

In [ ]:
# Display the dimensions
print("The credit rating dataset has", df.shape[0], "records, each with", df.shape[1],
    "attributes")

In [ ]:
# Display the structure
df.info()

We have 26 columns of numerical data and 6 descriptive columns (one of which is the label).There are no missing values.
A first look at the data:

#### Analyse Labels

As we know we are working with ordinal labels. That means there is a scale from more secure to less secure ratings. For instance, the triple-A (AAA) is the most secure rating a company can receive. On the other hand, the rating D is the less secure. It means the company will likely default on its creditors. Let's have a first look at the how many reatings we have of each in the dataset.

In [ ]:
df['Rating'].value_counts()

In [ ]:
df.groupby(['Rating Agency Name', 'Rating']).size()

We observe that the dataset is very unbalanced. We have 671 triple-Bs (BBB) but only 1 D. However, we are working with Ratings from different companies such as `Moody's`, `Standard & Poor's` and more. Therefore it is preferred to simplify the labels according to this table from the website [investopedia](https://www.investopedia.com/terms/c/corporate-credit-rating.asp). We will classify our labels according to the grading risk and not the rate. 

| Bond Rating |                   |          |            |              |
|-------------|-------------------|----------|------------|--------------|
| Moody's     | Standard & Poor's | Fitch    | Grade      | Risk         |
| Aaa         | AAA               | AAA      | Investment | Lowest Risk  |
| Aa          | AA                | AA       | Investment | Low Risk     |
| A           | A                 | A        | Investment | Low Risk     |
| Baa         | BBB               | BBB      | Investment | Medium Risk  |
| Ba, B       | BB, B             | BB, B    | Junk       | High Risk    |
| Caa/Ca      | CCC/CC/C          | CCC/CC/C | Junk       | Highest Risk |
| C           | D                 | D        | Junk       | In Default   |


To do it we will replace with a dictonary each of this ratings. 

In [ ]:
rating_dict = {'AAA':'Lowest Risk', 
               'AA':'Low Risk',
               'A':'Low Risk',
               'BBB':'Medium Risk', 
               'BB':'High Risk',
               'B':'High Risk',
               'CCC':'Highest Risk', 
               'CC':'Highest Risk',
               'C':'Highest Risk',
               'D':'In Default'}

df['Rating'] = df['Rating'].map(rating_dict)

In [ ]:
ax = df['Rating'].value_counts().plot(kind='bar',
                                        figsize=(8,4),
                                        title="Count of Rating by Type",
                                        grid=True)

While we can do multi-class prediction, to simplify for today's learning session, we will convert this into a binary problem:
**Risky Investment or not**
| Investment Type | Constitution       |Binary Representation|
|-----------------|--------------------|---------------------|
| Risky           | High Risk & above  |  1                  |
| Not Risky       | Medium Risk & below|  0                  |

In [ ]:
risk_type_dict = {'Lowest Risk': 0, 
               'Low Risk': 0,
                'Medium Risk': 0, 
                'High Risk': 1,
                'Highest Risk': 1,
                'In Default': 1}

df['Target'] = df['Rating'].map(risk_type_dict)

In [ ]:
ax = df['Target'].value_counts().plot(kind='bar',
                                        figsize=(8,4),
                                        title="Count of RiskType",
                                        grid=True)
ax.legend(['1: Risky & \n0: Not Risky']);

In [ ]:
column_list = list(df.columns[6:31])
# column_list = sample(column_list,4) 
print(column_list)

In [ ]:
for col in column_list:
    print(f'Column: {col}')
    sns.kdeplot(data=df[df['Target'] == 0], x=col);
    sns.kdeplot(data=df[df['Target'] == 1], x=col);
    plt.show();

In [ ]:
print('Checking outliers in data: ')
for c in column_list:

    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1 #Interquartile range
    fence_low  = q3 - 1.5 * iqr
    fence_high = q1 + 1.5 * iqr
    lower_out = len(df.loc[(df[c] < fence_low)  ,c])
    upper_out = len(df.loc[(df[c] > fence_high)  ,c])
    outlier_count = upper_out + lower_out
    prop_out = outlier_count / len(df)
    print(c, ": "+"{:.2%}".format(prop_out))


***If we were building a machine learning from scratch, depending on the type of model, we would have to treat the data including the outliers***

### 4️⃣ Understanding Model Selection Experiments ⚗️
Model selection is crucial. We need to compare different algorithms to determine the best performer.
Some approaches include:
- Logistic Regression
- Decision Trees
- Random Forest
- Gradient Boosting

Then based on the selected model, we need to apply data treatments.

### 5️⃣ Citizen Data Scientists 😎: The New Age Data Experts 
Citizen Data Scientists are business experts who leverage easy-to-use ML tools without deep technical expertise.

### 6️⃣ PyCaret: A Game-Changer for SMEs 🐍🥕
PyCaret simplifies machine learning tasks for non-programmers by automating feature selection, model tuning, and evaluation.

Read more here: https://pycaret.org/

In [ ]:
from pycaret.classification import *

In [ ]:
ignore_cols = [col for col in df.columns if col not in column_list and col != 'Target']

In [ ]:
exp1 = setup(data=df, target='Target',ignore_features=ignore_cols, session_id=42)

In [ ]:
# Compare different models
best_model = compare_models()

In [ ]:
# Finalize and save the model
final_model = finalize_model(best_model)
save_model(final_model, '../models/CCR_Prediction_Model')

In [ ]:
# Generate performance metrics
evaluate_model(best_model)

In [ ]:
# predict on test set
holdout_pred = predict_model(best_model)

In [ ]:
# show predictions df
holdout_pred.head()